# Phase 3 structured level-rate RF-QRC

Option B experiment.

This notebook tests whether separating level and rate qubits, then entangling mainly across the two blocks, reduces the calibration/tail tradeoff seen in the RF-QRC ring.

Qubit layout:

```text
q0, q1, q2 = level qubits
q3, q4, q5 = rate qubits
```

Variants:

```text
ring             = current RF-QRC control
cross_matched    = q0-q3, q1-q4, q2-q5
cross_all        = all level-rate pairs
block_plus_cross = weak within-block links + full cross links
```

Success criterion:

```text
repo-style QLIKE improves versus ring
and
q95 tail / crisis signal remains strong
```


In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if p.name == "qpitome-qrc-volatility" and (p / "scripts").exists() and (p / "src").exists():
            return p
    raise RuntimeError("Open this notebook from inside qpitome-qrc-volatility.")

ROOT = find_repo_root()
os.chdir(ROOT)

TABLES = ROOT / "results" / "tables"
FIGURES = ROOT / "results" / "figures"
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

print("Repo root:", ROOT)
print("Tables:", TABLES)
print("Figures:", FIGURES)


## Run structured RF-QRC script

In [ ]:
script = ROOT / "scripts" / "run_phase3_structured_level_rate_rf_qrc.py"
if not script.exists():
    raise FileNotFoundError(script)

cmd = [
    sys.executable,
    str(script),
    "--leak", "0.3",
    "--ridge-alpha", "3000",
    "--level-scale", "1.0",
    "--rate-scale", "0.75",
    "--cross-zz-boost", "1.0",
    "--weak-within-boost", "0.35",
    "--entanglers", "ring", "cross_matched", "cross_all", "block_plus_cross",
]

print(" ".join(cmd))
subprocess.run(cmd, cwd=ROOT, check=True)


## Load outputs

In [ ]:
metrics_path = TABLES / "phase3_structured_level_rate_rf_qrc_metrics.csv"
summary_path = TABLES / "phase3_structured_level_rate_rf_qrc_test_summary.csv"
pred_path = TABLES / "phase3_structured_level_rate_rf_qrc_predictions.csv"

metrics = pd.read_csv(metrics_path)
summary = pd.read_csv(summary_path)
preds = pd.read_csv(pred_path)

display(summary)
display(preds.head())


## Compare against current RF-QRC ring and Phase 2/ESN table if available

In [ ]:
# Optional context from the main clean comparison notebook.
context_path = TABLES / "phase3_clean_raw_forecast_comparison.csv"
if context_path.exists():
    context = pd.read_csv(context_path)
    display(context)
else:
    context = pd.DataFrame()
    print("No main comparison table found yet:", context_path)

structured = summary.copy()
structured["model"] = structured["entangler"].map({
    "ring": "Structured script: ring control",
    "cross_matched": "Structured: cross matched",
    "cross_all": "Structured: cross all",
    "block_plus_cross": "Structured: block + cross",
})
structured_view = structured[[
    "model", "rmse", "qlike", "corr", "actual_std", "pred_std",
    "q80_f1", "q90_f1", "q95_f1", "top20_pred_actual_ratio", "effective_rank"
]]

display(structured_view.sort_values("qlike"))


## Selection table

In [ ]:
selection = structured_view.copy()
ring_qlike = float(selection.loc[selection["model"].eq("Structured script: ring control"), "qlike"].iloc[0])
ring_q95 = float(selection.loc[selection["model"].eq("Structured script: ring control"), "q95_f1"].iloc[0])

selection["delta_qlike_vs_ring"] = selection["qlike"] - ring_qlike
selection["delta_q95_f1_vs_ring"] = selection["q95_f1"] - ring_q95
selection["passes_option_b_gate"] = (
    (selection["qlike"] < ring_qlike) &
    (selection["q95_f1"] >= 0.35)
)

selection_path = TABLES / "phase3_structured_level_rate_rf_qrc_selection_table.csv"
selection.to_csv(selection_path, index=False)
print("Saved:", selection_path)
display(selection.sort_values(["passes_option_b_gate", "qlike"], ascending=[False, True]))


## Save diagnostic figures

In [ ]:
def savefig(path: Path):
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    print("Saved:", path)
    plt.show()

plot = structured_view.set_index("model")

# Raw metrics.
cols = ["rmse", "qlike", "corr", "pred_std", "q95_f1", "top20_pred_actual_ratio"]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.ravel(), cols):
    plot[col].plot(kind="bar", ax=ax)
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=35)
plt.suptitle("Structured level-rate RF-QRC: raw diagnostic metrics")
savefig(FIGURES / "phase3_structured_level_rate_raw_metrics.png")

# QLIKE vs q95 F1 tradeoff.
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(selection["qlike"], selection["q95_f1"])
for _, row in selection.iterrows():
    ax.annotate(row["model"].replace("Structured: ", "").replace("Structured script: ", ""), 
                (row["qlike"], row["q95_f1"]), xytext=(5, 5), textcoords="offset points", fontsize=8)
ax.axhline(0.35, linestyle="--", linewidth=1)
ax.axvline(ring_qlike, linestyle="--", linewidth=1)
ax.set_xlabel("repo-style QLIKE (lower is better)")
ax.set_ylabel("q95 amplitude F1")
ax.set_title("Structured level-rate RF-QRC: QLIKE vs q95 tail F1")
savefig(FIGURES / "phase3_structured_level_rate_qlike_vs_q95_f1.png")

# Forecast traces for test period.
test = preds[preds["split"].astype(str).str.lower().eq("test")].copy()
x = pd.to_datetime(test["date"], errors="coerce")
if x.isna().all():
    x = np.arange(len(test))

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(x, test["actual_future_rv_20d"], label="actual", linewidth=2)
for col in [c for c in test.columns if c.endswith("_pred")]:
    label = col.replace("structured_level_rate_rf_qrc_", "").replace("_pred", "")
    ax.plot(x, test[col], label=label, alpha=0.85)
ax.set_title("Structured level-rate RF-QRC forecasts on test period")
ax.set_ylabel("future RV 20d")
ax.legend(ncol=3)
savefig(FIGURES / "phase3_structured_level_rate_test_forecasts.png")


## Interpretation checklist

In [ ]:
print("Option B gate:")
print("  qlike improves versus ring control")
print("  q95 F1 remains >= 0.35")
print()
display(selection[["model", "qlike", "q95_f1", "delta_qlike_vs_ring", "delta_q95_f1_vs_ring", "passes_option_b_gate"]])

best = selection.sort_values(["passes_option_b_gate", "qlike"], ascending=[False, True]).iloc[0]
print("\nBest candidate by gate then QLIKE:")
print(best.to_string())
